# 03 · Held-out test: TabPFN-3.5 vs NESO, 2025–2026

Same model as `02`, scored once on 2025-01-01 → 2026-09-15 under the protocol frozen in `docs/method.md`.
Monthly runs refit on every hour settled a day before issue; `frozen` runs keep the 2024 context throughout.
`blend` is the fixed 50/50 WINDFOR/power-curve benchmark (power curve fitted on Mar–Jul 2024), the bar for `tabpfn+windfor`.

In [ ]:
import pandas as pd
from windpfn import baselines, dataset, figures, model

d = baselines.add(dataset.build("2024-03-16", "2026-09-15").set_index("valid_time"))
X = model.features(d)
F = {"tabpfn": X, "tabpfn+windfor": X.join(d.windfor)}

In [ ]:
start = "2025-01-01"
runs = {k + ("" if refit else " frozen"): model.backtest(x, d, start, refit=refit) for k, x in F.items() for refit in (True, False)}
test = d[start:].assign(**runs)
err = test[["windfor", "blend", *runs]].sub(test.y, axis=0)
model.score(err)

In [ ]:
figures.monthly(err);

In [ ]:
err.abs().groupby(pd.cut(test.lead_h, [15, 21, 27, 33, 40]), observed=True).mean().round(0)